# AKT

Student-blocked nested evaluation of AKT on two binary prediction tasks:

- **First attempt:** `firstattemptcorrect`
- **After feedback:** `eventualcorrect`

Run the setup cell, then either task cell. Each task uses its own fixed outer-fold file and output directory. Hyperparameters and the final training epoch count are selected exclusively from inner validation folds.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the interaction-level FeedBook export.
DATA_PATH = Path("data/feedbook_interactions.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Update DATA_PATH before running."
    )

logging_data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(logging_data):,} interactions from {DATA_PATH}")


## First-attempt prediction


In [ ]:
# AKT First Attempt

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, mean_absolute_error
)
import optuna
from optuna.samplers import GridSampler
from statistics import mean, stdev

# ==== IMPORT AKT FROM PYKT ====
from pykt.models import akt
AKT = akt.AKT

# -------------------------
# Reproducibility & device
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Parameters
# -------------------------
batch_size = 8
n_splits = 3
max_epochs = 100
patience = 10

# -------------------------
# Model/KC identifiers
# -------------------------
model_name = "AKT"
kc_name = "itemid"   # change per run
question_col = "KC (itemid)"   # change per run
correct_col = "firstattemptcorrect"
student_col = "Anon Student Id"

# -------------------------
# Load your data
# -------------------------
# logging_data = pd.read_csv("your_file.csv")

# IMPORTANT:
# Create row_id on the full original dataset BEFORE model-specific filtering,
# so all models can be aligned later.
logging_data = logging_data.copy().reset_index(drop=True)
if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data))

# -------------------------
# Fixed outer folds
# -------------------------
fixed_fold_file = "data/fixed_outer_folds.csv"

if os.path.exists(fixed_fold_file):
    fixed_folds = pd.read_csv(fixed_fold_file)
    logging_data = logging_data.merge(fixed_folds, on=["row_id"], how="left")
    if logging_data["outer_fold"].isna().any():
        raise ValueError("Some row_id values are missing outer_fold assignments.")
    logging_data["outer_fold"] = logging_data["outer_fold"].astype(int)
    print(f"Loaded fixed folds from {fixed_fold_file}")
else:
    # Build folds once on the FULL dataset using student groups
    # Only rows with valid student ids can participate in fold assignment
    base_for_folds = logging_data.dropna(subset=[student_col]).copy()

    gkf = GroupKFold(n_splits=n_splits)
    base_for_folds["outer_fold"] = -1

    for fold_idx, (_, test_idx) in enumerate(
        gkf.split(
            base_for_folds,
            groups=base_for_folds[student_col]
        ),
        start=1
    ):
        base_for_folds.iloc[
            test_idx,
            base_for_folds.columns.get_loc("outer_fold")
        ] = fold_idx

    fixed_folds = base_for_folds[["row_id", "outer_fold"]].copy()
    fixed_folds.to_csv(fixed_fold_file, index=False)
    logging_data = logging_data.merge(
        fixed_folds,
        on="row_id",
        how="left"
    )

    if logging_data["outer_fold"].isna().any():
        # rows lacking student ids cannot be used anyway and will be dropped below
        pass

    print(f"Saved fixed folds to {fixed_fold_file}")

# -------------------------
# Model-specific preprocessing
# -------------------------
logging_model = logging_data.dropna(
    subset=[
        question_col,
        correct_col,
        student_col,
        "outer_fold"
    ]
).copy()

logging_model[correct_col] = logging_model[correct_col].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

# Map every original row_id to its student ID so the OOF files
# can support student-level or clustered statistical tests later.
student_id_lookup = (
    logging_model[
        ["row_id", student_col]
    ]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[student_col]
    .to_dict()
)

# --- Sequence length ---
seq_len = 20
print(f"seq_len: {seq_len}")

# Global question ID mapping for THIS KC column
all_qids = logging_model[question_col].dropna().unique()

qid_to_index = {
    qid: idx + 1
    for idx, qid in enumerate(all_qids)
}  # 0 reserved for PAD

num_questions = len(qid_to_index) + 1

# -------------------------
# Dataset
# -------------------------
class KTDataFromLogging(Dataset):
    def __init__(
        self,
        df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index
    ):
        self.seq_len = seq_len
        self.samples = []

        df = df.copy()

        df["qid_index"] = (
            df[question_col]
            .map(qid_to_index)
            .fillna(0)
            .astype(int)
        )

        # Important: preserve original row order within each student's sequence.
        # If you have a timestamp/order column, sort here before grouping/chunking.
        # Example:
        # df = df.sort_values([student_col, "timestamp"])

        for _, group in df.groupby(student_col, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[correct_col].astype(int).tolist()
            row_seq = group["row_id"].tolist()

            for start in range(0, len(q_seq), seq_len - 1):
                end = min(start + seq_len, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]

                pad_len = seq_len - len(q_chunk)

                if pad_len > 0:
                    q_chunk += [0] * pad_len
                    r_chunk += [0] * pad_len
                    row_chunk += [-1] * pad_len

                self.samples.append(
                    (
                        torch.tensor(
                            q_chunk,
                            dtype=torch.long
                        ),
                        torch.tensor(
                            r_chunk,
                            dtype=torch.long
                        ),
                        torch.tensor(
                            row_chunk,
                            dtype=torch.long
                        )
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# -------------------------
# Hyperparameter grid
# -------------------------
param_search_space = {
    "d_model": [64, 128, 256],
    "d_ff": [64, 128, 256],
    "dropout": [0.1, 0.3, 0.5],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "num_attn_heads": [4, 8],
    "n_blocks": [1, 2, 4],
}

# -------------------------
# Helpers
# -------------------------
def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )
    )

# -------------------------
# Storage
# -------------------------
# Save all OOF prediction files inside AE_FA.
# Create the folder automatically if it does not already exist.
oof_output_dir = "AKT_FA"
os.makedirs(oof_output_dir, exist_ok=True)

all_preds_all_folds = []
all_labels_all_folds = []
all_rowids_all_folds = []
all_foldnums_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []

# -------------------------
# Outer CV using fixed folds
# -------------------------
for fold in range(1, n_splits + 1):
    print(f"\n=== Outer Fold {fold}/{n_splits} ===")

    train_val_df = logging_model[
        logging_model["outer_fold"] != fold
    ].copy()

    test_df = logging_model[
        logging_model["outer_fold"] == fold
    ].copy()

    # Inner CV for hyperparameter tuning
    inner_cv = GroupKFold(n_splits=n_splits)
    inner_groups = train_val_df[student_col]

    def objective(trial):
        d_model = trial.suggest_categorical(
            "d_model",
            param_search_space["d_model"]
        )

        d_ff = trial.suggest_categorical(
            "d_ff",
            param_search_space["d_ff"]
        )

        lr = trial.suggest_categorical(
            "learning_rate",
            param_search_space["learning_rate"]
        )

        num_attn_heads = trial.suggest_categorical(
            "num_attn_heads",
            param_search_space["num_attn_heads"]
        )

        dropout = trial.suggest_categorical(
            "dropout",
            param_search_space["dropout"]
        )

        n_blocks = trial.suggest_categorical(
            "n_blocks",
            param_search_space["n_blocks"]
        )

        auc_scores = []
        best_epochs = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_val_df,
            groups=inner_groups
        ):
            inner_train_df = train_val_df.iloc[inner_train_idx]
            inner_val_df = train_val_df.iloc[inner_val_idx]

            train_dataset = KTDataFromLogging(
                inner_train_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index
            )

            val_dataset = KTDataFromLogging(
                inner_val_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False
            )

            model = AKT(
                n_question=num_questions,
                n_pid=0,
                d_model=d_model,
                n_blocks=n_blocks,
                dropout=dropout,
                d_ff=d_ff,
                num_attn_heads=num_attn_heads,
                emb_type="qid"
            ).to(device)

            criterion = nn.BCELoss()

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=lr
            )

            best_auc_inner = -np.inf
            best_epoch_inner = 1
            no_improve = 0

            for epoch in range(max_epochs):
                model.train()

                for q_batch, r_batch, rowid_batch in train_loader:
                    q_batch, r_batch = (
                        q_batch.to(device),
                        r_batch.to(device)
                    )

                    optimizer.zero_grad()

                    # AKT loss with the regularization term
                    outputs, c_reg_loss = model(
                        q_batch,
                        r_batch
                    )

                    valid_mask = q_batch[:, 1:] != 0

                    loss = criterion(
                        outputs[:, 1:][valid_mask],
                        r_batch[:, 1:].float()[valid_mask]
                    ) + c_reg_loss

                    loss.backward()
                    optimizer.step()

                # Validation
                model.eval()
                val_preds, val_labels = [], []

                with torch.no_grad():
                    for q_batch, r_batch, rowid_batch in val_loader:
                        q_batch, r_batch = (
                            q_batch.to(device),
                            r_batch.to(device)
                        )

                        outputs, _ = model(
                            q_batch,
                            r_batch
                        )

                        for b in range(q_batch.size(0)):
                            for t in range(seq_len - 1):
                                next_q = q_batch[b, t + 1]

                                if next_q == 0:
                                    continue

                                val_preds.append(
                                    outputs[b, t + 1].item()
                                )

                                val_labels.append(
                                    r_batch[b, t + 1].item()
                                )

                if (
                    len(val_labels) > 0
                    and len(np.unique(val_labels)) > 1
                ):
                    val_auc = roc_auc_score(
                        val_labels,
                        val_preds
                    )

                    if val_auc > best_auc_inner:
                        best_auc_inner = val_auc
                        best_epoch_inner = epoch + 1
                        no_improve = 0
                    else:
                        no_improve += 1

                    if no_improve >= patience:
                        break

            auc_scores.append(best_auc_inner)
            best_epochs.append(best_epoch_inner)

        trial.set_user_attr(
            "recommended_epochs",
            max(1, int(np.median(best_epochs))),
        )

        return np.mean(auc_scores) if auc_scores else 0.0

    grid_size = 1

    for _, v in param_search_space.items():
        grid_size *= len(v)

    sampler = GridSampler(param_search_space)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    study.optimize(
        objective,
        n_trials=grid_size,
        show_progress_bar=True
    )

    best_params = study.best_params
    recommended_epochs = int(
        study.best_trial.user_attrs["recommended_epochs"]
    )

    print(
        f"\nBest params for fold {fold}: "
        f"{best_params}, AUC: {study.best_value:.6f}"
    )

    # -------------------------
    # Retrain best model on train_val_df
    # -------------------------
    train_dataset = KTDataFromLogging(
        train_val_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index
    )

    test_dataset = KTDataFromLogging(
        test_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    best_model = AKT(
        n_question=num_questions,
        n_pid=0,
        d_model=best_params["d_model"],
        n_blocks=best_params["n_blocks"],
        dropout=best_params["dropout"],
        d_ff=best_params["d_ff"],
        num_attn_heads=best_params["num_attn_heads"],
        emb_type="qid"
    ).to(device)

    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"]
    )

    # Retrain on every outer-training student for the epoch count
    # selected from the inner validation folds.
    for epoch in range(recommended_epochs):
        best_model.train()

        for q_batch, r_batch, rowid_batch in train_loader:
            q_batch, r_batch = (
                q_batch.to(device),
                r_batch.to(device)
            )

            optimizer.zero_grad()

            # AKT loss with the regularization term
            outputs, c_reg_loss = best_model(
                q_batch,
                r_batch
            )

            valid_mask = q_batch[:, 1:] != 0

            loss = criterion(
                outputs[:, 1:][valid_mask],
                r_batch[:, 1:].float()[valid_mask]
            ) + c_reg_loss

            loss.backward()
            optimizer.step()

        # The epoch count is selected only from inner validation folds.
        # The outer test fold is not inspected during training.

    # Final evaluation + save row-level OOF predictions
    # -------------------------
    best_model.eval()
    fold_preds, fold_labels, fold_row_ids = [], [], []

    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in test_loader:
            q_batch, r_batch = (
                q_batch.to(device),
                r_batch.to(device)
            )

            outputs, _ = best_model(
                q_batch,
                r_batch
            )

            for b in range(q_batch.size(0)):
                for t in range(seq_len - 1):
                    next_q = q_batch[b, t + 1]

                    if next_q == 0:
                        continue

                    row_id = rowid_batch[b, t + 1].item()

                    if row_id == -1:
                        continue

                    pred = outputs[b, t + 1].item()
                    label = r_batch[b, t + 1].item()

                    fold_preds.append(pred)
                    fold_labels.append(label)
                    fold_row_ids.append(row_id)

                    all_preds_all_folds.append(pred)
                    all_labels_all_folds.append(label)
                    all_rowids_all_folds.append(row_id)
                    all_foldnums_all_folds.append(fold)

    if len(fold_labels) > 0:
        binary_preds = [
            1 if p > 0.5 else 0
            for p in fold_preds
        ]

        auc = (
            roc_auc_score(
                fold_labels,
                fold_preds
            )
            if len(np.unique(fold_labels)) > 1
            else np.nan
        )

        acc = accuracy_score(
            fold_labels,
            binary_preds
        )

        rmse = rmse_score(
            fold_labels,
            fold_preds
        )

        mae = mean_absolute_error(
            fold_labels,
            fold_preds
        )

        precision = precision_score(
            fold_labels,
            binary_preds,
            zero_division=0
        )

        recall = recall_score(
            fold_labels,
            binary_preds,
            zero_division=0
        )

        f1 = f1_score(
            fold_labels,
            binary_preds,
            zero_division=0
        )

        auc_per_fold.append(auc)
        acc_per_fold.append(acc)
        rmse_per_fold.append(rmse)
        mae_per_fold.append(mae)
        precision_per_fold.append(precision)
        recall_per_fold.append(recall)
        f1_per_fold.append(f1)

        print(f"\nEvaluation on Fold {fold} Test Set:")
        print(f"AUC: {auc:.6f}")
        print(f"Accuracy: {acc:.6f}")
        print(f"RMSE: {rmse:.6f}")
        print(f"MAE: {mae:.6f}")
        print(f"Precision: {precision:.6f}")
        print(f"Recall: {recall:.6f}")
        print(f"F1 Score: {f1:.6f}")

        fold_df = pd.DataFrame({
            "row_id": fold_row_ids,
            "student_id": [
                student_id_lookup[row_id]
                for row_id in fold_row_ids
            ],
            "fold": fold,
            "y_true": fold_labels,
            "y_pred": fold_preds,
            "model_name": model_name,
            "kc_name": kc_name,
        })

        fold_outfile = os.path.join(
            oof_output_dir,
            f"oof_{model_name}_{kc_name}_fold{fold}.csv"
        )

        fold_df.to_csv(
            fold_outfile,
            index=False
        )

        print(
            f"Saved Fold {fold} OOF predictions to "
            f"{fold_outfile}"
        )

# -------------------------
# Save combined OOF file
# -------------------------
oof_df = pd.DataFrame({
    "row_id": all_rowids_all_folds,
    "student_id": [
        student_id_lookup[row_id]
        for row_id in all_rowids_all_folds
    ],
    "fold": all_foldnums_all_folds,
    "y_true": all_labels_all_folds,
    "y_pred": all_preds_all_folds,
    "model_name": model_name,
    "kc_name": kc_name,
}).sort_values(
    "row_id"
).reset_index(
    drop=True
)

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv"
)

oof_df.to_csv(
    oof_outfile,
    index=False
)

print(f"\nSaved combined OOF predictions to {oof_outfile}")

print("\n=== Final Averaged Results (3-Fold CV) ===")

print(
    f"Mean AUC: "
    f"{mean(auc_per_fold):.6f} "
    f"± {stdev(auc_per_fold):.6f}"
    if len(auc_per_fold) > 1
    else
    f"Mean AUC: {mean(auc_per_fold):.6f}"
)

print(f"Mean Accuracy: {mean(acc_per_fold):.6f}")
print(f"Mean RMSE: {mean(rmse_per_fold):.6f}")
print(f"Mean MAE: {mean(mae_per_fold):.6f}")
print(f"Mean Precision: {mean(precision_per_fold):.6f}")
print(f"Mean Recall: {mean(recall_per_fold):.6f}")
print(f"Mean F1:  {mean(f1_per_fold):.6f}")

## After-feedback prediction


In [ ]:
# AKT After Feedback

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, mean_absolute_error
)
import optuna
from optuna.samplers import GridSampler
from statistics import mean, stdev

# ==== IMPORT AKT FROM PYKT ====
from pykt.models import akt
AKT = akt.AKT

# -------------------------
# Reproducibility & device
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Parameters
# -------------------------
batch_size = 8
n_splits = 3
max_epochs = 100
patience = 10

# -------------------------
# Model/KC identifiers
# -------------------------
model_name = "AKT"
kc_name = "itemid"   # change per run
question_col = "KC (itemid)"   # change per run
correct_col = "eventualcorrect"
student_col = "Anon Student Id"

# -------------------------
# Load your data
# -------------------------
# logging_data = pd.read_csv("your_file.csv")

# IMPORTANT:
# Create row_id on the full original dataset BEFORE model-specific filtering,
# so all models can be aligned later.
logging_data = logging_data.copy().reset_index(drop=True)
if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data))

# -------------------------
# Fixed outer folds
# -------------------------
fixed_fold_file = "data/fixed_outer_folds_af.csv"

if os.path.exists(fixed_fold_file):
    fixed_folds = pd.read_csv(fixed_fold_file)
    logging_data = logging_data.merge(fixed_folds, on=["row_id"], how="left")
    if logging_data["outer_fold"].isna().any():
        raise ValueError("Some row_id values are missing outer_fold assignments.")
    logging_data["outer_fold"] = logging_data["outer_fold"].astype(int)
    print(f"Loaded fixed folds from {fixed_fold_file}")
else:
    # Build folds once on the FULL dataset using student groups
    # Only rows with valid student ids can participate in fold assignment
    base_for_folds = logging_data.dropna(subset=[student_col]).copy()

    gkf = GroupKFold(n_splits=n_splits)
    base_for_folds["outer_fold"] = -1

    for fold_idx, (_, test_idx) in enumerate(
        gkf.split(
            base_for_folds,
            groups=base_for_folds[student_col]
        ),
        start=1
    ):
        base_for_folds.iloc[
            test_idx,
            base_for_folds.columns.get_loc("outer_fold")
        ] = fold_idx

    fixed_folds = base_for_folds[["row_id", "outer_fold"]].copy()
    fixed_folds.to_csv(fixed_fold_file, index=False)
    logging_data = logging_data.merge(
        fixed_folds,
        on="row_id",
        how="left"
    )

    if logging_data["outer_fold"].isna().any():
        # rows lacking student ids cannot be used anyway and will be dropped below
        pass

    print(f"Saved fixed folds to {fixed_fold_file}")

# -------------------------
# Model-specific preprocessing
# -------------------------
logging_model = logging_data.dropna(
    subset=[
        question_col,
        correct_col,
        student_col,
        "outer_fold"
    ]
).copy()

logging_model[correct_col] = logging_model[correct_col].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

# Map every original row_id to its student ID so the OOF files
# can support student-level or clustered statistical tests later.
student_id_lookup = (
    logging_model[
        ["row_id", student_col]
    ]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[student_col]
    .to_dict()
)

# --- Sequence length ---
seq_len = 20
print(f"seq_len: {seq_len}")

# Global question ID mapping for THIS KC column
all_qids = logging_model[question_col].dropna().unique()

qid_to_index = {
    qid: idx + 1
    for idx, qid in enumerate(all_qids)
}  # 0 reserved for PAD

num_questions = len(qid_to_index) + 1

# -------------------------
# Dataset
# -------------------------
class KTDataFromLogging(Dataset):
    def __init__(
        self,
        df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index
    ):
        self.seq_len = seq_len
        self.samples = []

        df = df.copy()

        df["qid_index"] = (
            df[question_col]
            .map(qid_to_index)
            .fillna(0)
            .astype(int)
        )

        # Important: preserve original row order within each student's sequence.
        # If you have a timestamp/order column, sort here before grouping/chunking.
        # Example:
        # df = df.sort_values([student_col, "timestamp"])

        for _, group in df.groupby(student_col, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[correct_col].astype(int).tolist()
            row_seq = group["row_id"].tolist()

            for start in range(0, len(q_seq), seq_len - 1):
                end = min(start + seq_len, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]

                pad_len = seq_len - len(q_chunk)

                if pad_len > 0:
                    q_chunk += [0] * pad_len
                    r_chunk += [0] * pad_len
                    row_chunk += [-1] * pad_len

                self.samples.append(
                    (
                        torch.tensor(
                            q_chunk,
                            dtype=torch.long
                        ),
                        torch.tensor(
                            r_chunk,
                            dtype=torch.long
                        ),
                        torch.tensor(
                            row_chunk,
                            dtype=torch.long
                        )
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# -------------------------
# Hyperparameter grid
# -------------------------
param_search_space = {
    "d_model": [64, 128, 256],
    "d_ff": [64, 128, 256],
    "dropout": [0.1, 0.3, 0.5],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "num_attn_heads": [4, 8],
    "n_blocks": [1, 2, 4],
}

# -------------------------
# Helpers
# -------------------------
def rmse_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )
    )

# -------------------------
# Storage
# -------------------------
# Save all OOF prediction files inside AE_FA.
# Create the folder automatically if it does not already exist.
oof_output_dir = "AKT_AF"
os.makedirs(oof_output_dir, exist_ok=True)

all_preds_all_folds = []
all_labels_all_folds = []
all_rowids_all_folds = []
all_foldnums_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []

# -------------------------
# Outer CV using fixed folds
# -------------------------
for fold in range(1, n_splits + 1):
    print(f"\n=== Outer Fold {fold}/{n_splits} ===")

    train_val_df = logging_model[
        logging_model["outer_fold"] != fold
    ].copy()

    test_df = logging_model[
        logging_model["outer_fold"] == fold
    ].copy()

    # Inner CV for hyperparameter tuning
    inner_cv = GroupKFold(n_splits=n_splits)
    inner_groups = train_val_df[student_col]

    def objective(trial):
        d_model = trial.suggest_categorical(
            "d_model",
            param_search_space["d_model"]
        )

        d_ff = trial.suggest_categorical(
            "d_ff",
            param_search_space["d_ff"]
        )

        lr = trial.suggest_categorical(
            "learning_rate",
            param_search_space["learning_rate"]
        )

        num_attn_heads = trial.suggest_categorical(
            "num_attn_heads",
            param_search_space["num_attn_heads"]
        )

        dropout = trial.suggest_categorical(
            "dropout",
            param_search_space["dropout"]
        )

        n_blocks = trial.suggest_categorical(
            "n_blocks",
            param_search_space["n_blocks"]
        )

        auc_scores = []
        best_epochs = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_val_df,
            groups=inner_groups
        ):
            inner_train_df = train_val_df.iloc[inner_train_idx]
            inner_val_df = train_val_df.iloc[inner_val_idx]

            train_dataset = KTDataFromLogging(
                inner_train_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index
            )

            val_dataset = KTDataFromLogging(
                inner_val_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True
            )

            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False
            )

            model = AKT(
                n_question=num_questions,
                n_pid=0,
                d_model=d_model,
                n_blocks=n_blocks,
                dropout=dropout,
                d_ff=d_ff,
                num_attn_heads=num_attn_heads,
                emb_type="qid"
            ).to(device)

            criterion = nn.BCELoss()

            optimizer = torch.optim.Adam(
                model.parameters(),
                lr=lr
            )

            best_auc_inner = -np.inf
            best_epoch_inner = 1
            no_improve = 0

            for epoch in range(max_epochs):
                model.train()

                for q_batch, r_batch, rowid_batch in train_loader:
                    q_batch, r_batch = (
                        q_batch.to(device),
                        r_batch.to(device)
                    )

                    optimizer.zero_grad()

                    # AKT loss with the regularization term
                    outputs, c_reg_loss = model(
                        q_batch,
                        r_batch
                    )

                    valid_mask = q_batch[:, 1:] != 0

                    loss = criterion(
                        outputs[:, 1:][valid_mask],
                        r_batch[:, 1:].float()[valid_mask]
                    ) + c_reg_loss

                    loss.backward()
                    optimizer.step()

                # Validation
                model.eval()
                val_preds, val_labels = [], []

                with torch.no_grad():
                    for q_batch, r_batch, rowid_batch in val_loader:
                        q_batch, r_batch = (
                            q_batch.to(device),
                            r_batch.to(device)
                        )

                        outputs, _ = model(
                            q_batch,
                            r_batch
                        )

                        for b in range(q_batch.size(0)):
                            for t in range(seq_len - 1):
                                next_q = q_batch[b, t + 1]

                                if next_q == 0:
                                    continue

                                val_preds.append(
                                    outputs[b, t + 1].item()
                                )

                                val_labels.append(
                                    r_batch[b, t + 1].item()
                                )

                if (
                    len(val_labels) > 0
                    and len(np.unique(val_labels)) > 1
                ):
                    val_auc = roc_auc_score(
                        val_labels,
                        val_preds
                    )

                    if val_auc > best_auc_inner:
                        best_auc_inner = val_auc
                        best_epoch_inner = epoch + 1
                        no_improve = 0
                    else:
                        no_improve += 1

                    if no_improve >= patience:
                        break

            auc_scores.append(best_auc_inner)
            best_epochs.append(best_epoch_inner)

        trial.set_user_attr(
            "recommended_epochs",
            max(1, int(np.median(best_epochs))),
        )

        return np.mean(auc_scores) if auc_scores else 0.0

    grid_size = 1

    for _, v in param_search_space.items():
        grid_size *= len(v)

    sampler = GridSampler(param_search_space)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    study.optimize(
        objective,
        n_trials=grid_size,
        show_progress_bar=True
    )

    best_params = study.best_params
    recommended_epochs = int(
        study.best_trial.user_attrs["recommended_epochs"]
    )

    print(
        f"\nBest params for fold {fold}: "
        f"{best_params}, AUC: {study.best_value:.6f}"
    )

    # -------------------------
    # Retrain best model on train_val_df
    # -------------------------
    train_dataset = KTDataFromLogging(
        train_val_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index
    )

    test_dataset = KTDataFromLogging(
        test_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    best_model = AKT(
        n_question=num_questions,
        n_pid=0,
        d_model=best_params["d_model"],
        n_blocks=best_params["n_blocks"],
        dropout=best_params["dropout"],
        d_ff=best_params["d_ff"],
        num_attn_heads=best_params["num_attn_heads"],
        emb_type="qid"
    ).to(device)

    criterion = nn.BCELoss()

    optimizer = torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"]
    )

    # Retrain on every outer-training student for the epoch count
    # selected from the inner validation folds.
    for epoch in range(recommended_epochs):
        best_model.train()

        for q_batch, r_batch, rowid_batch in train_loader:
            q_batch, r_batch = (
                q_batch.to(device),
                r_batch.to(device)
            )

            optimizer.zero_grad()

            # AKT loss with the regularization term
            outputs, c_reg_loss = best_model(
                q_batch,
                r_batch
            )

            valid_mask = q_batch[:, 1:] != 0

            loss = criterion(
                outputs[:, 1:][valid_mask],
                r_batch[:, 1:].float()[valid_mask]
            ) + c_reg_loss

            loss.backward()
            optimizer.step()

        # The epoch count is selected only from inner validation folds.
        # The outer test fold is not inspected during training.

    # Final evaluation + save row-level OOF predictions
    # -------------------------
    best_model.eval()
    fold_preds, fold_labels, fold_row_ids = [], [], []

    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in test_loader:
            q_batch, r_batch = (
                q_batch.to(device),
                r_batch.to(device)
            )

            outputs, _ = best_model(
                q_batch,
                r_batch
            )

            for b in range(q_batch.size(0)):
                for t in range(seq_len - 1):
                    next_q = q_batch[b, t + 1]

                    if next_q == 0:
                        continue

                    row_id = rowid_batch[b, t + 1].item()

                    if row_id == -1:
                        continue

                    pred = outputs[b, t + 1].item()
                    label = r_batch[b, t + 1].item()

                    fold_preds.append(pred)
                    fold_labels.append(label)
                    fold_row_ids.append(row_id)

                    all_preds_all_folds.append(pred)
                    all_labels_all_folds.append(label)
                    all_rowids_all_folds.append(row_id)
                    all_foldnums_all_folds.append(fold)

    if len(fold_labels) > 0:
        binary_preds = [
            1 if p > 0.5 else 0
            for p in fold_preds
        ]

        auc = (
            roc_auc_score(
                fold_labels,
                fold_preds
            )
            if len(np.unique(fold_labels)) > 1
            else np.nan
        )

        acc = accuracy_score(
            fold_labels,
            binary_preds
        )

        rmse = rmse_score(
            fold_labels,
            fold_preds
        )

        mae = mean_absolute_error(
            fold_labels,
            fold_preds
        )

        precision = precision_score(
            fold_labels,
            binary_preds,
            zero_division=0
        )

        recall = recall_score(
            fold_labels,
            binary_preds,
            zero_division=0
        )

        f1 = f1_score(
            fold_labels,
            binary_preds,
            zero_division=0
        )

        auc_per_fold.append(auc)
        acc_per_fold.append(acc)
        rmse_per_fold.append(rmse)
        mae_per_fold.append(mae)
        precision_per_fold.append(precision)
        recall_per_fold.append(recall)
        f1_per_fold.append(f1)

        print(f"\nEvaluation on Fold {fold} Test Set:")
        print(f"AUC: {auc:.6f}")
        print(f"Accuracy: {acc:.6f}")
        print(f"RMSE: {rmse:.6f}")
        print(f"MAE: {mae:.6f}")
        print(f"Precision: {precision:.6f}")
        print(f"Recall: {recall:.6f}")
        print(f"F1 Score: {f1:.6f}")

        fold_df = pd.DataFrame({
            "row_id": fold_row_ids,
            "student_id": [
                student_id_lookup[row_id]
                for row_id in fold_row_ids
            ],
            "fold": fold,
            "y_true": fold_labels,
            "y_pred": fold_preds,
            "model_name": model_name,
            "kc_name": kc_name,
        })

        fold_outfile = os.path.join(
            oof_output_dir,
            f"oof_{model_name}_{kc_name}_fold{fold}.csv"
        )

        fold_df.to_csv(
            fold_outfile,
            index=False
        )

        print(
            f"Saved Fold {fold} OOF predictions to "
            f"{fold_outfile}"
        )

# -------------------------
# Save combined OOF file
# -------------------------
oof_df = pd.DataFrame({
    "row_id": all_rowids_all_folds,
    "student_id": [
        student_id_lookup[row_id]
        for row_id in all_rowids_all_folds
    ],
    "fold": all_foldnums_all_folds,
    "y_true": all_labels_all_folds,
    "y_pred": all_preds_all_folds,
    "model_name": model_name,
    "kc_name": kc_name,
}).sort_values(
    "row_id"
).reset_index(
    drop=True
)

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv"
)

oof_df.to_csv(
    oof_outfile,
    index=False
)

print(f"\nSaved combined OOF predictions to {oof_outfile}")

print("\n=== Final Averaged Results (3-Fold CV) ===")

print(
    f"Mean AUC: "
    f"{mean(auc_per_fold):.6f} "
    f"± {stdev(auc_per_fold):.6f}"
    if len(auc_per_fold) > 1
    else
    f"Mean AUC: {mean(auc_per_fold):.6f}"
)

print(f"Mean Accuracy: {mean(acc_per_fold):.6f}")
print(f"Mean RMSE: {mean(rmse_per_fold):.6f}")
print(f"Mean MAE: {mean(mae_per_fold):.6f}")
print(f"Mean Precision: {mean(precision_per_fold):.6f}")
print(f"Mean Recall: {mean(recall_per_fold):.6f}")
print(f"Mean F1:  {mean(f1_per_fold):.6f}")